# Component 1 Walkthrough â€” GIK Parquet â†’ IceChunk Virtual Store

C1 ingests GIK Parquet key-reference files produced by ECMWF IFS ensemble output
and commits them as virtual Zarr arrays into an IceChunk store with full time-travel support.

**Key classes**: `GIKFlatParquetParser`, `IceChainStore`

In [ ]:
import numpy as np
import xarray as xr
import pandas as pd
from pathlib import Path
import json, tempfile
from datetime import date

WORK_DIR = Path(tempfile.mkdtemp(prefix='gik_c1_'))
print('Working directory:', WORK_DIR)

NLAT, NLON, NMEMBERS, NSTEPS = 8, 8, 10, 16
LAT   = np.linspace(0.0, 4.0, NLAT, dtype=np.float32)
LON   = np.linspace(35.0, 39.0, NLON, dtype=np.float32)
STEPS = np.arange(0, NSTEPS * 6, 6, dtype=np.int32)
TEST_DATE = date(2024, 10, 15)

## 1.1  GIK Parquet Schema

The GIK Parquet format stores chunk references in a flat key-value table:

| key | value |
|-----|-------|
| `step_NNN/{var}/sfc/{member}/{chunk_idx}` | `["s3://bucket/file.grib2", offset, length]` |
| `{var}/{level_type}/{level}/.zarray` | `{ zarr metadata JSON }` |

In [ ]:
rows = []
for step in STEPS:
    rows.append({
        'key':   f'step_{step}/tp/sfc/0/0',
        'value': json.dumps([f's3://ecmwf-forecasts/fake/{step}.grib2', int(step) * 100, 500]),
    })
rows.append({
    'key': 'tp/heightAboveGround/0/.zarray',
    'value': json.dumps({
        'chunks': [1, NLAT, NLON], 'compressor': None, 'dtype': '<f4',
        'fill_value': 'NaN', 'filters': None, 'order': 'C',
        'shape': [NSTEPS, NLAT, NLON], 'zarr_format': 2,
    }),
})

parquet_path = WORK_DIR / 'sample.parquet'
pd.DataFrame(rows).to_parquet(parquet_path)
pd.read_parquet(parquet_path)

## 1.2  Parsing with GIKFlatParquetParser

The parser extracts step-hour integers and maps chunk refs into a VirtualiZarr ManifestStore.

In [ ]:
import re

_SFC_STEP_PATTERN = re.compile(r'^step_(\d+)/')

df = pd.read_parquet(parquet_path)
df["key"] = df["key"].astype(str)
step_hours = sorted(
    int(m.group(1))
    for k in df["key"]
    if (m := _SFC_STEP_PATTERN.match(k))
)
step_hours = sorted(set(step_hours))
print(f"SFC step hours found: {step_hours}")
assert step_hours == sorted(int(s) for s in STEPS)
print("Step hours correctly extracted from Parquet — OK")

## 1.3  IceChunk Store â€” Create, Commit, Tag

In [ ]:
try:
    from gik_icechain.conversion.icechunk_writer import IceChainStore

    store_path = str(WORK_DIR / 'icechunk_store')
    store = IceChainStore(store_path)
    store.create_or_open()

    synthetic_ds = xr.Dataset({'tp': xr.DataArray(
        np.random.default_rng(0).random((NMEMBERS, NSTEPS, NLAT, NLON)).astype(np.float32),
        dims=['member', 'step', 'latitude', 'longitude'],
        coords={
            'member':    np.arange(NMEMBERS),
            'step':      STEPS,
            'latitude':  LAT,
            'longitude': LON,
        },
        attrs={'units': 'm'},
    )})

    commit_hash = store.commit_day(TEST_DATE, synthetic_ds, run_hour=0)
    print(f'Committed snapshot: {commit_hash[:12]}...')
    STORE_OK = True
except ImportError as e:
    print(f'Skipping: {e}')
    STORE_OK = False

## 1.4  Time-Travel Checkout

`checkout_as_of(date)` reads the snapshot tagged for that date.

In [ ]:
if STORE_OK:
    historical = store.checkout_as_of(TEST_DATE)
    print(historical)
    assert 'tp' in historical.data_vars
    print('Time-travel checkout â€” OK')
else:
    print('Skipping â€” store not initialised')

## 1.5  Snapshots & Validation

In [ ]:
if STORE_OK:
    snapshots = store.list_snapshots()
    for s in snapshots:
        print(s)

    report = store.validate()
    print(report)
    assert report['committed_days'] >= 1
    assert report['gaps_detected'] == 0
    print('Validation passed â€” OK')
else:
    print('Skipping â€” store not initialised')